# On-Device Voice Agent — Colab Runner (RQ2 first real number)

This notebook gets your project into Colab, installs CPU inference runtimes,
downloads **one** small GGUF model, and runs the **RQ2 latency-split** experiment
for real — your fastest path to a citable result (testing the refuted "TTS
dominates" claim).

**Runtime:** CPU is fine (this is the whole point). `Runtime > Change runtime type > CPU`.

Steps: (1) get code → (2) install → (3) download model → (4) sanity dummy run →
(5) real RQ2 → (6) plot the breakdown.

## 1. Get the code into Colab
Pick **one** option, run it, leave the others commented.

In [ ]:
# OPTION A — clone from your GitHub repo (recommended once you've pushed):
REPO_URL = ""  # e.g. "https://github.com/yourname/voice-agent-quant.git"
import os
if REPO_URL:
    !git clone $REPO_URL project
    %cd project

# OPTION B — mount Google Drive where you've copied the project folder:
# from google.colab import drive; drive.mount('/content/drive')
# %cd /content/drive/MyDrive/research   # <- adjust path

# OPTION C — upload a zip of the project, then:
# from google.colab import files; up = files.upload()
# !unzip -q list(up)[0] -d project && cd project

!ls -1  # you should see src/, requirements.txt, scripts/

## 2. Install dependencies
Core + model runtimes. `llama-cpp-python` builds a CPU wheel (~1-2 min).

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -r requirements-models.txt
print('deps installed')

## 3. Download one small model
We use **Qwen2.5-3B-Instruct Q4_K_M** for a fast first run (the paper target is
7B; swap the repo/filename when you scale up). faster-whisper + Piper voices
download on first use / below.

In [ ]:
import os
from huggingface_hub import hf_hub_download
os.makedirs('models', exist_ok=True)

# Target LLM (Q4_K_M, ~2 GB)
llm_path = hf_hub_download(
    repo_id='bartowski/Qwen2.5-3B-Instruct-GGUF',
    filename='Qwen2.5-3B-Instruct-Q4_K_M.gguf',
    local_dir='models')
print('LLM:', llm_path)

# Piper voice (small)
voice = hf_hub_download(
    repo_id='rhasspy/piper-voices',
    filename='en/en_US/amy/medium/en_US-amy-medium.onnx',
    local_dir='models')
voice_cfg = hf_hub_download(
    repo_id='rhasspy/piper-voices',
    filename='en/en_US/amy/medium/en_US-amy-medium.onnx.json',
    local_dir='models')
os.environ['VA_PIPER_VOICE'] = voice
os.environ['VA_ASR_MODEL'] = 'base'   # small whisper for speed on Colab CPU
print('Piper voice:', voice)

## 4. Sanity check — dummy run (no models, instant)

In [ ]:
!python -m src.experiments.rq2_latency --dummy --n-turns 3

## 5. Real RQ2 latency split
Real ASR (faster-whisper) + LLM (llama.cpp, CPU) + TTS (Piper). First call is
slower (model load + whisper download). `--model-path` points at the GGUF we
downloaded, overriding `config.py` so you don't have to edit it.

In [ ]:
import os
llm_path = 'models/Qwen2.5-3B-Instruct-Q4_K_M.gguf'
!python -m src.experiments.rq2_latency --quant Q4_K_M --model-path $llm_path --n-turns 5

## 6. Plot the latency breakdown
Load the results JSONL and visualize which stage dominates — your RQ2 figure.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, glob
f = sorted(glob.glob('results/rq2_latency_*.jsonl'))[-1]
df = pd.read_json(f, lines=True)
stage_means = df[['asr_s','llm_decode_s','tts_total_s']].mean() * 1000  # ms
ax = stage_means.plot.bar(color=['#4C72B0','#C44E52','#55A868'])
ax.set_ylabel('mean latency (ms)'); ax.set_title(f'RQ2 latency split ({f.split("_")[-1]})')
for i,v in enumerate(stage_means): ax.text(i, v, f'{v:.0f}', ha='center', va='bottom')
plt.tight_layout(); plt.show()
dom = stage_means.idxmax()
print(f'Dominant stage: {dom} ({stage_means[dom]/stage_means.sum()*100:.0f}% of stage time)')
print('LLM-dominated → supports our hypothesis, contra the refuted "TTS dominates" claim.' 
      if dom=='llm_decode_s' else 'Unexpected — investigate.')

## Next
- Sweep quant levels: download `Q8_0, Q5_K_M, Q3_K_M, Q2_K` too and loop the run
  cell over `--quant`/`--model-path` to get the latency-vs-quant curve.
- Then move to **RQ1** (`rq1_quality.py`) once real BFCL/MT-Bench data is wired,
  and **RQ3** (`rq3_specdec.py`) with a 0.5B draft model.
- Cache `models/` to Drive so you don't re-download each session.